# RANSAC Empirical Analysis
**CS5008 Final Project — Arsh Singh**

This notebook runs all three experiments and generates plots for the report.

---
## 1. Setup and Imports

In [3]:
import sys
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# output directory for saved figures
import os
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)
DATA_DIR = '../results'
os.makedirs(DATA_DIR, exist_ok=True)

# Column rename dictionary shared across all three experiments.
# Apply with: df.rename(columns=COL_NAMES, inplace=True)

COL_NAMES = {
    'index':       'Run Index',
    'n':           'Total Points (N)',
    'epsilon':     'Outlier Fraction (ε)',
    'd':           'Expected Inliers (d)',
    'm':           'Model Parameters (m)',
    'k':           'Iterations Used (k)',
    'k_budget':    'Iteration Budget (k)',
    'k_theory':    'Iterations Required by Theory',
    't':           'Inlier Threshold (t)',
    'pr':          'Bias Probability (pr)',
    'bias_type':   'Bias Type',
    'repeat':      'Repeat Number',
    'time_mu_s':   'Wall-Clock Time (μs)',
    'model_error': 'Model Error (Euclidean)',
}
print('Setup complete.')

Setup complete.


---
## Experiment 1 — How Does RANSAC Break Down as Outlier Fraction Increases?

Fix $k$ at the value computed for $\varepsilon = 0.5$. Vary $\varepsilon$ from 0.1 to 0.9.
Run for linear ($m=2$) and quadratic ($m=3$) models. `N` is 1000. Repeat `N_REPEATS` times.

In [4]:
df1 = pd.read_csv('results/exp1.csv').rename(columns=COL_NAMES)
df1.head()

,Run Index,Total Points (N),Outlier Fraction (ε),Expected Inliers (d),Model Parameters (m),Iterations Used (k),Inlier Threshold (t),Repeat Number,Wall-Clock Time (μs),Model Error (Euclidean)
0,0,1000,0.05,950,2,17,0.9992,0,397.0,0.112070
1,1,1000,0.05,950,2,17,1.0011,1,349.0,0.090577
2,2,1000,0.05,950,2,17,0.9663,2,390.0,0.190570
3,3,1000,0.05,950,2,17,0.9853,3,340.0,0.089918
4,4,1000,0.05,950,2,17,1.0075,4,392.0,0.075750


In [11]:
agg1 = (df1.groupby(['Model Parameters (m)', 'Outlier Fraction (ε)'])
           ['Model Error (Euclidean)']
           .agg(mean_error='mean', std_error='std', count='count')
           .reset_index())

In [12]:
agg1 = (df1.groupby(['Model Parameters (m)', 'Outlier Fraction (ε)'])
           ['Model Error (Euclidean)']
           .agg(mean_error='mean', std_error='std', count='count')
           .reset_index())

# 95% confidence interval half-width: 1.96 * std / sqrt(n)
agg1['ci95'] = 1.96 * agg1['std_error'] / np.sqrt(agg1['count'])
agg1['ci_lo'] = agg1['mean_error'] - agg1['ci95']
agg1['ci_hi'] = agg1['mean_error'] + agg1['ci95']


KeyError: 'mean'

In [ ]:
# breakdown threshold — model error greater than twice noise std
BREAKDOWN_THRESHOLD = 2 * 0.5   # 2 * NOISE_STD

---
## Experiment 2 — At What Structural Bias Probability Does RANSAC Fail?

Fix $\varepsilon = 0.3$. Vary $p_r$ from 0.0 to 1.0 in steps of 0.1.
Three bias types: constant, linear, periodic. Linear and quadratic models. `N` is 1000. Repeat `N_REPEATS` times.

---
## Experiment 3 — At What Model Degree Does RANSAC Fail for a Budgeted or Fixed Iteration Count?

Fix $\varepsilon = 0.1$. Vary $m$ from 2 to 11 in steps of 1. Run twice for $k$ = 100 and $k$ = 1000. `N` is 1000. Repeat `N_REPEATS` times.
